# LeetCode 2529 — Maximum Count of Positive Integer and Negative Integer bằng Python

Link đề: https://leetcode.com/problems/maximum-count-of-positive-integer-and-negative-integer/ · Độ khó: **Easy** · Chủ đề: Array, Binary Search, Counting

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def maximumCount(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số phần tử **âm** và số phần tử **dương** của một mảng đã **sắp xếp không giảm**, rồi lấy **số lớn hơn** (số `0` không tính vào bên nào).
2. Biết đếm tuyến tính bằng vòng lặp/`sum` và nhận ra khi mảng đã sắp xếp có thể dùng **tìm kiếm nhị phân**.
3. Hiểu `bisect_left` và `bisect_right` trả về vị trí chèn trái/phải, và cách dùng chúng để biết **bao nhiêu phần tử `< 0`** và **bao nhiêu phần tử `> 0`**.
4. Viết 4 cách: đếm bằng vòng lặp, `sum` với điều kiện, `bisect`, và tự viết tìm kiếm nhị phân (lower bound); đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: tính số `0` vào số dương, cộng hai số đếm thay vì lấy `max`, sai biên khi tự viết tìm kiếm nhị phân, và bỏ sót trường hợp toàn số âm/toàn số dương/toàn số `0`.

## 2. Đề bài & ví dụ

Cho mảng `nums` đã được **sắp xếp theo thứ tự không giảm**. Trả về **giá trị lớn hơn** giữa:

- số lượng số nguyên **dương** trong `nums`;
- số lượng số nguyên **âm** trong `nums`.

Lưu ý: số `0` **không** phải số dương cũng không phải số âm.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[-2,-1,-1,1,2,3]` | `3` | có 3 số dương và 3 số âm; lớn hơn là `3` |
| 2 | `[-3,-2,-1,0,0,1,2]` | `3` | có 2 số dương và 3 số âm; lớn hơn là `3` |
| 3 | `[5,20,66,1314]` | `4` | có 4 số dương và 0 số âm; lớn hơn là `4` |

Constraints: `1 <= nums.length <= 2000`; `-2000 <= nums[i] <= 2000`; `nums` được sắp xếp theo thứ tự không giảm.

Follow-up: có thể giải trong `O(log n)` không?

In [1]:
examples = [([-2, -1, -1, 1, 2, 3], 3), ([-3, -2, -1, 0, 0, 1, 2], 3), ([5, 20, 66, 1314], 4)]
for nums, expected in examples:
    neg = sum(1 for x in nums if x < 0)
    pos = sum(1 for x in nums if x > 0)
    print(f"nums={nums}: âm={neg}, dương={pos} -> max = {max(neg, pos)} (mong đợi {expected})")

nums=[-2, -1, -1, 1, 2, 3]: âm=3, dương=3 -> max = 3 (mong đợi 3)
nums=[-3, -2, -1, 0, 0, 1, 2]: âm=3, dương=2 -> max = 3 (mong đợi 3)
nums=[5, 20, 66, 1314]: âm=0, dương=4 -> max = 4 (mong đợi 4)


## 3. Kiến thức nền

### 3.1. Mảng sắp xếp không giảm có dạng "âm - không - dương"

Vì mảng không giảm, các số âm đứng **đầu**, rồi tới các số `0`, rồi các số dương ở **cuối**. Ba nhóm nằm thành ba đoạn liên tiếp, không xen kẽ nhau.

In [2]:
nums = [-3, -2, -1, 0, 0, 1, 2]
print("âm  :", [x for x in nums if x < 0])
print("zero:", [x for x in nums if x == 0])
print("dương:", [x for x in nums if x > 0])

âm  : [-3, -2, -1]
zero: [0, 0]
dương: [1, 2]


### 3.2. Đếm bằng `sum` với điều kiện

`sum(1 for x in nums if x < 0)` đếm số phần tử thỏa điều kiện. Cũng có thể viết `sum(x < 0 for x in nums)` vì `True` bằng `1` khi cộng.

In [3]:
nums = [-3, -2, -1, 0, 0, 1, 2]
print(sum(1 for x in nums if x < 0), sum(x > 0 for x in nums))

3 2


### 3.3. `bisect_left` và `bisect_right`

Với mảng đã sắp xếp `a` và giá trị `x`:

- `bisect_left(a, x)` = vị trí chèn **bên trái** các phần tử bằng `x` = **số phần tử `< x`**.
- `bisect_right(a, x)` = vị trí chèn **bên phải** các phần tử bằng `x` = **số phần tử `<= x`**.

Từ đó: số âm = `bisect_left(nums, 0)`; số dương = `len(nums) - bisect_right(nums, 0)`.

In [4]:
from bisect import bisect_left, bisect_right
a = [-3, -2, -1, 0, 0, 1, 2]
print("bisect_left(a, 0)  =", bisect_left(a, 0), "(số phần tử < 0)")
print("bisect_right(a, 0) =", bisect_right(a, 0), "(số phần tử <= 0)")
print("số dương =", len(a) - bisect_right(a, 0))

bisect_left(a, 0)  = 3 (số phần tử < 0)
bisect_right(a, 0) = 5 (số phần tử <= 0)
số dương = 2


### 3.4. Tìm kiếm nhị phân tự viết (lower bound)

Lower bound của `x`: chỉ số nhỏ nhất `i` sao cho `a[i] >= x` (hoặc `len(a)` nếu không có). Khuôn mẫu: `lo, hi = 0, len(a)`; trong khi `lo < hi` đặt `mid = (lo + hi) // 2`; nếu `a[mid] < x` thì `lo = mid + 1`, ngược lại `hi = mid`. Trả về `lo`.

In [5]:
def lower_bound(a, x):
    lo, hi = 0, len(a)              # khoảng [lo, hi) nửa mở
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

a = [-3, -2, -1, 0, 0, 1, 2]
print(lower_bound(a, 0), lower_bound(a, 1), lower_bound(a, -5), lower_bound(a, 100))

3 5 0 7


### 3.5. `max` của hai số

`max(neg, pos)` trả về số lớn hơn. Đề yêu cầu **max**, không phải tổng.

In [6]:
print(max(3, 2), max(0, 4))

3 4


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `[-3,-2,-1,0,0,1,2]` (độ dài 7):

- Có 3 số `< 0` (ba số đầu), 2 số `0`, 2 số `> 0`.
- `max(3, 2) = 3`.

Nhờ mảng đã sắp xếp, ta chỉ cần **hai ranh giới**:

- Ranh giới 1: vị trí số `>= 0` đầu tiên là chỉ số `3` nên có `3` số âm.
- Ranh giới 2: vị trí số `> 0` đầu tiên là chỉ số `5` nên có `7 - 5 = 2` số dương.

Hai ranh giới này có thể tìm bằng tìm kiếm nhị phân thay vì đếm từng phần tử.

In [7]:
a = [-3, -2, -1, 0, 0, 1, 2]
first_nonneg = next(i for i, x in enumerate(a) if x >= 0)
first_pos = next((i for i, x in enumerate(a) if x > 0), len(a))
print("chỉ số số >= 0 đầu tiên:", first_nonneg, "-> số âm =", first_nonneg)
print("chỉ số số > 0 đầu tiên:", first_pos, "-> số dương =", len(a) - first_pos)

chỉ số số >= 0 đầu tiên: 3 -> số âm = 3
chỉ số số > 0 đầu tiên: 5 -> số dương = 2


## 5. Cách 1 — Brute Force / baseline: đếm bằng vòng lặp

**Ý tưởng:** duyệt từng phần tử, tăng `neg` nếu `x < 0`, tăng `pos` nếu `x > 0`, bỏ qua `0`. Trả về `max(neg, pos)`.

In [8]:
from typing import List

class SolutionBrute:
    def maximumCount(self, nums: List[int]) -> int:
        neg = pos = 0
        for x in nums:
            if x < 0:
                neg += 1
            elif x > 0:
                pos += 1
        return max(neg, pos)

In [9]:
sol = SolutionBrute()
print(sol.maximumCount([-2, -1, -1, 1, 2, 3]))        # 3
print(sol.maximumCount([-3, -2, -1, 0, 0, 1, 2]))     # 3
print(sol.maximumCount([5, 20, 66, 1314]))            # 4

3
3
4


**Độ phức tạp:** Time `O(n)`, Space `O(1)`. **Nút thắt:** không tận dụng tính chất đã sắp xếp; với bài này `n <= 2000` thì vẫn nhanh, nhưng follow-up hỏi `O(log n)`.

## 6. Ý tưởng thuật toán / bước đột phá

Mảng đã sắp xếp có dạng `[âm..., 0..., dương...]`, nên:

- **số âm** = chỉ số đầu tiên của phần tử `>= 0` = `bisect_left(nums, 0)`;
- **số dương** = `n -` chỉ số đầu tiên của phần tử `> 0` = `n - bisect_right(nums, 0)`.

Mỗi giá trị tìm bằng một lần tìm kiếm nhị phân `O(log n)`. Không cần xét số `0` riêng vì chúng nằm giữa hai ranh giới và bị loại tự nhiên.

**Quy tắc:** *`neg = bisect_left(nums, 0)`; `pos = n - bisect_right(nums, 0)`; đáp án `max(neg, pos)`.*

**Invariant của tìm kiếm nhị phân:** `lo` luôn chỉ vào vị trí mà mọi phần tử ở bên trái `lo` đều `< x`, và mọi phần tử từ `hi` trở đi đều `>= x`.

## 7. Các cách tiếp theo

### Cách 2 — `sum` với điều kiện (hai dòng, vẫn `O(n)`)

In [10]:
from typing import List

class SolutionSum:
    def maximumCount(self, nums: List[int]) -> int:
        neg = sum(1 for x in nums if x < 0)
        pos = sum(1 for x in nums if x > 0)
        return max(neg, pos)

In [11]:
sol = SolutionSum()
print(sol.maximumCount([-2, -1, -1, 1, 2, 3]))        # 3
print(sol.maximumCount([-3, -2, -1, 0, 0, 1, 2]))     # 3
print(sol.maximumCount([5, 20, 66, 1314]))            # 4

3
3
4


**Độ phức tạp:** Time `O(n)` (hai lần duyệt), Space `O(1)`.

### Cách 3 — Thư viện `bisect`

Dùng `bisect_left` và `bisect_right` có sẵn của Python, đạt `O(log n)`.

In [12]:
from bisect import bisect_left, bisect_right
from typing import List

class SolutionBisect:
    def maximumCount(self, nums: List[int]) -> int:
        neg = bisect_left(nums, 0)                    # số phần tử < 0
        pos = len(nums) - bisect_right(nums, 0)       # số phần tử > 0
        return max(neg, pos)

In [13]:
sol = SolutionBisect()
print(sol.maximumCount([-2, -1, -1, 1, 2, 3]))        # 3
print(sol.maximumCount([-3, -2, -1, 0, 0, 1, 2]))     # 3
print(sol.maximumCount([5, 20, 66, 1314]))            # 4

3
3
4


**Độ phức tạp:** Time `O(log n)`, Space `O(1)`.

### Cách 4 — Tự viết tìm kiếm nhị phân

Dùng hàm `lower_bound(a, x)` = số phần tử `< x`. Với số nguyên: số phần tử `<= 0` bằng số phần tử `< 1`, nên `pos = n - lower_bound(nums, 1)`.

In [14]:
from typing import List

class SolutionBinary:
    def maximumCount(self, nums: List[int]) -> int:
        def lower_bound(x: int) -> int:               # số phần tử < x (mảng đã sắp xếp)
            lo, hi = 0, len(nums)
            while lo < hi:
                mid = (lo + hi) // 2
                if nums[mid] < x:
                    lo = mid + 1
                else:
                    hi = mid
            return lo
        neg = lower_bound(0)                          # phần tử < 0
        pos = len(nums) - lower_bound(1)              # phần tử >= 1, tức > 0 (số nguyên)
        return max(neg, pos)

In [15]:
sol = SolutionBinary()
print(sol.maximumCount([-2, -1, -1, 1, 2, 3]))        # 3
print(sol.maximumCount([-3, -2, -1, 0, 0, 1, 2]))     # 3
print(sol.maximumCount([5, 20, 66, 1314]))            # 4

3
3
4


**Độ phức tạp:** Time `O(log n)`, Space `O(1)`. Mẹo `lower_bound(1)`: vì `nums` toàn số nguyên nên "`> 0`" tương đương "`>= 1`".

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [-3,-2,-1,0,0,1,2]` (`n = 7`), tìm `lower_bound(0)`:

| Bước | `lo` | `hi` | `mid` | `nums[mid]` | Quyết định |
|---|---|---|---|---|---|
| 1 | 0 | 7 | 3 | 0 | `0 < 0`? không nên `hi = 3` |
| 2 | 0 | 3 | 1 | -2 | `-2 < 0` nên `lo = 2` |
| 3 | 2 | 3 | 2 | -1 | `-1 < 0` nên `lo = 3` |
| 4 | 3 | 3 | | | `lo == hi`, trả `3` |

Vậy `neg = 3`. Tương tự `lower_bound(1) = 5` nên `pos = 7 - 5 = 2`; đáp án `max(3, 2) = 3`.

In [16]:
def trace_lower_bound(nums, x):
    lo, hi = 0, len(nums)
    print(f"lower_bound(x={x}) trên {nums}")
    while lo < hi:
        mid = (lo + hi) // 2
        go_right = nums[mid] < x
        print(f"  lo={lo}, hi={hi}, mid={mid}, nums[mid]={nums[mid]}, nums[mid] < {x}? {go_right}")
        if go_right:
            lo = mid + 1
        else:
            hi = mid
    print(f"  -> lo = hi = {lo}")
    return lo

nums = [-3, -2, -1, 0, 0, 1, 2]
neg = trace_lower_bound(nums, 0)
pos = len(nums) - trace_lower_bound(nums, 1)
print("neg =", neg, "| pos =", pos, "| đáp án =", max(neg, pos))

lower_bound(x=0) trên [-3, -2, -1, 0, 0, 1, 2]
  lo=0, hi=7, mid=3, nums[mid]=0, nums[mid] < 0? False
  lo=0, hi=3, mid=1, nums[mid]=-2, nums[mid] < 0? True
  lo=2, hi=3, mid=2, nums[mid]=-1, nums[mid] < 0? True
  -> lo = hi = 3
lower_bound(x=1) trên [-3, -2, -1, 0, 0, 1, 2]
  lo=0, hi=7, mid=3, nums[mid]=0, nums[mid] < 1? True
  lo=4, hi=7, mid=5, nums[mid]=1, nums[mid] < 1? False
  lo=4, hi=5, mid=4, nums[mid]=0, nums[mid] < 1? True
  -> lo = hi = 5
neg = 3 | pos = 2 | đáp án = 3


## 9. Code nộp LeetCode

Tìm kiếm nhị phân bằng `bisect` (`O(log n)`):

In [17]:
from bisect import bisect_left, bisect_right
from typing import List

class Solution:
    def maximumCount(self, nums: List[int]) -> int:
        neg = bisect_left(nums, 0)
        pos = len(nums) - bisect_right(nums, 0)
        return max(neg, pos)

In [18]:
print(Solution().maximumCount([-2, -1, -1, 1, 2, 3]))
print(Solution().maximumCount([-3, -2, -1, 0, 0, 1, 2]))
print(Solution().maximumCount([5, 20, 66, 1314]))

3
3
4


## 10. Vì sao thuật toán đúng

1. **Cấu trúc của mảng sắp xếp.** Với mảng không giảm, mọi số `< 0` đứng trước mọi số `0`, và mọi số `0` đứng trước mọi số `> 0`. Do đó ba nhóm là ba đoạn liên tiếp.
2. **Đếm số âm.** `bisect_left(nums, 0)` là chỉ số đầu tiên `i` có `nums[i] >= 0`. Mọi phần tử ở chỉ số `< i` đều `< 0`, và có đúng `i` phần tử như vậy.
3. **Đếm số dương.** `bisect_right(nums, 0)` là chỉ số đầu tiên có `nums[i] > 0`. Mọi phần tử từ chỉ số đó đến cuối đều `> 0`, số lượng `n - bisect_right`.
4. **Số `0` bị loại tự nhiên.** Chúng nằm trong đoạn `[bisect_left, bisect_right)` không thuộc hai đếm.
5. **Tính đúng của tìm kiếm nhị phân.** Invariant: mọi phần tử trước `lo` đều `< x`, mọi phần tử từ `hi` trở đi đều `>= x`. Mỗi vòng thu hẹp `[lo, hi)` còn một nửa; kết thúc khi `lo == hi` thì đó chính là ranh giới.
6. **Trường hợp biên.** Mảng toàn âm: `bisect_left = n`, `pos = 0`. Toàn dương: `neg = 0`, `pos = n`. Toàn `0`: cả hai bằng `0`. Các trường hợp này đều đúng vì công thức không phân biệt.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp | `O(n)` | `O(1)` |
| 2. `sum` điều kiện | `O(n)` | `O(1)` |
| 3. `bisect` | `O(log n)` | `O(1)` |
| 4. Tự viết nhị phân | `O(log n)` | `O(1)` |

`n = len(nums) <= 2000`. Cách `O(log n)` đáp ứng follow-up; tuy nhiên với `n` nhỏ thì cách `O(n)` cũng chạy tức thì.

## 12. Bộ test

### 12.1. Ví dụ của đề, edge case và phản ví dụ

Mỗi kỳ vọng đều tính tay (ghi chú bên cạnh).

In [19]:
sols = [SolutionBrute(), SolutionSum(), SolutionBisect(), SolutionBinary(), Solution()]
cases = [
    ([-2, -1, -1, 1, 2, 3], 3),                 # ví dụ 1
    ([-3, -2, -1, 0, 0, 1, 2], 3),              # ví dụ 2
    ([5, 20, 66, 1314], 4),                     # ví dụ 3
    ([0], 0),                                   # chỉ một số 0
    ([0, 0, 0], 0),                             # toàn số 0
    ([-1], 1),                                  # một số âm
    ([1], 1),                                   # một số dương
    ([-5, -4, -3], 3),                          # toàn âm
    ([-1, 0, 1], 1),                            # cân bằng, có số 0 ở giữa
    ([-2, -1, 0, 0, 0, 3], 2),                  # âm 2, dương 1
    ([-1, -1, 0, 2, 2, 2], 3),                  # âm 2, dương 3
    ([0, 0, 5], 1),                             # chỉ dương, đầu là 0
    ([-7, 0], 1),                               # chỉ âm, cuối là 0
    ([-2000, 2000], 1),                         # hai biên
    ([-2000] * 1000 + [2000] * 1000, 1000),     # độ dài tối đa 2000, cân bằng
    ([0] * 2000, 0),                            # độ dài tối đa, toàn 0
]
for nums, want in cases:
    for s in sols:
        got = s.maximumCount(nums[:])
        assert got == want, (type(s).__name__, nums[:10], got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 16


Phản ví dụ cho lỗi "tính `0` vào số dương" và lỗi "cộng thay vì `max`": `[-3,-2,-1,0,0,1,2]` nếu tính `0` là dương thì dương `= 4 > 3` sai; nếu cộng thì ra `5`.

In [20]:
nums = [-3, -2, -1, 0, 0, 1, 2]
assert Solution().maximumCount(nums) == 3
assert len(nums) - bisect_left(nums, 0) == 4 and 4 != 3           # nếu tính 0 là dương thì ra 4, sai
assert bisect_left(nums, 0) + (len(nums) - bisect_right(nums, 0)) == 5 and 5 != 3
print("phản ví dụ OK")

phản ví dụ OK


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh `nums` đúng constraints: độ dài `1..2000`, giá trị trong `[-2000, 2000]`, **đã sắp xếp không giảm** (sinh rồi `sort`). Dùng nhiều dạng: trải đều, nhiều số `0`, toàn âm, toàn dương.

In [21]:
import random
random.seed(2529)

def gen(kind):
    n = random.randint(1, random.choice([5, 30, 300, 2000]))
    if kind == 0:
        arr = [random.randint(-2000, 2000) for _ in range(n)]
    elif kind == 1:
        arr = [random.choice([-3, -2, -1, 0, 0, 0, 1, 2, 3]) for _ in range(n)]
    elif kind == 2:
        arr = [random.randint(-2000, -1) for _ in range(n)]
    elif kind == 3:
        arr = [random.randint(1, 2000) for _ in range(n)]
    else:
        arr = [random.randint(-5, 5) for _ in range(n)]
    arr.sort()
    return arr

def oracle(nums):
    neg = sum(1 for x in nums if x < 0)
    pos = sum(1 for x in nums if x > 0)
    return max(neg, pos)

for t in range(3000):
    nums = gen(t % 5)
    assert 1 <= len(nums) <= 2000 and all(-2000 <= x <= 2000 for x in nums)
    assert nums == sorted(nums)
    ans = [s.maximumCount(nums[:]) for s in sols]
    assert len(set(ans)) == 1 and ans[0] == oracle(nums), (nums[:10], ans)
print("3000 test ngẫu nhiên: 5 cách khớp nhau và khớp oracle")

3000 test ngẫu nhiên: 5 cách khớp nhau và khớp oracle


### 12.3. Tính chất của đáp án

`ceil(số phần tử khác 0 / 2) <= đáp án <= số phần tử khác 0`. Đáp án bằng `0` khi và chỉ khi mảng toàn `0`. Đảo dấu và đảo mảng (vẫn đã sắp xếp) không đổi đáp án.

In [22]:
random.seed(9)
for t in range(1500):
    nums = gen(t % 5)
    ans = Solution().maximumCount(nums)
    nonzero = sum(1 for x in nums if x != 0)
    assert (nonzero + 1) // 2 <= ans <= nonzero
    assert (ans == 0) == (nonzero == 0)
    mirrored = sorted(-x for x in nums)                  # đảo dấu rồi sắp xếp lại
    assert Solution().maximumCount(mirrored) == ans
print("các tính chất chặn và đối xứng đều đúng")

các tính chất chặn và đối xứng đều đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tính số `0` vào nhóm số dương

In [23]:
def sai_tinh_zero(nums):
    neg = bisect_left(nums, 0)
    pos = len(nums) - bisect_left(nums, 0)               # SAI: gồm cả số 0
    return max(neg, pos)

nums = [-3, -2, -1, 0, 0, 1, 2]
print("đúng:", Solution().maximumCount(nums), "| sai:", sai_tinh_zero(nums))
assert Solution().maximumCount(nums) != sai_tinh_zero(nums)

đúng: 3 | sai: 4


**Vì sao sai:** `bisect_left(nums, 0)` đứng **trước** các số `0`, nên `n - bisect_left` gồm cả số `0` và số dương (4 số). Phải dùng `bisect_right` để đứng **sau** các số `0`.

### Lỗi 2: Cộng hai số đếm thay vì lấy `max`

In [24]:
def sai_cong(nums):
    neg = bisect_left(nums, 0)
    pos = len(nums) - bisect_right(nums, 0)
    return neg + pos                                      # SAI: đề yêu cầu số lớn hơn

nums = [-2, -1, -1, 1, 2, 3]
print("đúng:", Solution().maximumCount(nums), "| sai:", sai_cong(nums))
assert Solution().maximumCount(nums) != sai_cong(nums)

đúng: 3 | sai: 6


**Vì sao sai:** đề hỏi **giá trị lớn hơn** giữa số dương và số âm, không phải tổng. Với ví dụ 1 tổng `6`, đáp án `3`.

### Lỗi 3: Chỉ đếm số âm, quên số dương

In [25]:
def sai_chi_dem_am(nums):
    return bisect_left(nums, 0)                           # SAI: quên so với số dương

nums = [5, 20, 66, 1314]
print("đúng:", Solution().maximumCount(nums), "| sai:", sai_chi_dem_am(nums))
assert Solution().maximumCount(nums) != sai_chi_dem_am(nums)

đúng: 4 | sai: 0


**Vì sao sai:** mảng toàn số dương cho `neg = 0` nên bỏ `pos` là sai.

### Lỗi 4: Sai biên khi tự viết tìm kiếm nhị phân (`hi = n - 1` với vòng lặp `lo < hi`)

In [26]:
def sai_bien(nums, x):
    lo, hi = 0, len(nums) - 1                             # SAI: hi phải là len(nums) để có thể trả về n
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo

nums = [-5, -4, -3]
print("số âm đúng:", bisect_left(nums, 0), "| sai:", sai_bien(nums, 0))
assert bisect_left(nums, 0) != sai_bien(nums, 0)

số âm đúng: 3 | sai: 2


**Vì sao sai:** nếu **mọi** phần tử đều `< x` thì vị trí chèn là `n`, nhưng với `hi = n - 1` kết quả không thể vượt `n - 1`. Khuôn mẫu đúng cho "khoảng nửa mở" là `lo, hi = 0, n`.

### Lỗi 5: Dùng `lower_bound(0)` cho số dương thay vì `lower_bound(1)`

In [27]:
def sai_lb0(nums):
    def lb(x):
        lo, hi = 0, len(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] < x:
                lo = mid + 1
            else:
                hi = mid
        return lo
    return max(lb(0), len(nums) - lb(0))                  # SAI: lb(0) cũng đứng trước số 0

nums = [-3, -2, -1, 0, 0, 1, 2]
print("đúng:", Solution().maximumCount(nums), "| sai:", sai_lb0(nums))
assert Solution().maximumCount(nums) != sai_lb0(nums)

đúng: 3 | sai: 4


**Cách sửa:** số dương là `n - lb(1)` (với số nguyên), hoặc dùng `bisect_right(nums, 0)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mảng đã sắp xếp" + "đếm số phần tử thỏa điều kiện theo ngưỡng" ⇒ **tìm kiếm nhị phân / `bisect`**.
- `bisect_left(a, x)` = số phần tử `< x`; `bisect_right(a, x)` = số phần tử `<= x`. Học thuộc hai câu này.
- Số `0` nằm giữa: số âm = `bisect_left(0)`, số dương = `n - bisect_right(0)`.
- Với `n <= 2000`, cách đếm `O(n)` cũng được; nhưng follow-up luôn hỏi `O(log n)`.
- Xử lý biên: toàn âm, toàn dương, toàn `0`.
- Một dòng: `max(bisect_left(nums, 0), len(nums) - bisect_right(nums, 0))`.

In [28]:
def one_line(nums):
    return max(bisect_left(nums, 0), len(nums) - bisect_right(nums, 0))

assert one_line([-2, -1, -1, 1, 2, 3]) == 3 and one_line([5, 20, 66, 1314]) == 4
print("phiên bản một dòng đúng")

phiên bản một dòng đúng


## 15. Biến thể của bài

1. **Mảng chưa sắp xếp:** không dùng được nhị phân, quay lại đếm `O(n)`.
2. **Số lần xuất hiện của một giá trị `x` trong mảng sắp xếp:** `bisect_right(x) - bisect_left(x)`.
3. **Đếm phần tử nằm trong đoạn `[lo, hi]`:** `bisect_right(hi) - bisect_left(lo)`.
4. **Chỉ số của số dương nhỏ nhất:** `bisect_right(nums, 0)` nếu còn phần tử.
5. **Số cặp `(i, j)` có tổng âm** (sắp xếp + hai con trỏ).
6. **Phiên bản trên ma trận sắp xếp:** đếm số âm trong ma trận (LeetCode 1351).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp | đếm `neg`, `pos` | `O(n)` | `O(1)` | dễ hiểu nhất, mảng chưa sắp xếp |
| 2. `sum` điều kiện | hai lần đếm gọn | `O(n)` | `O(1)` | code ngắn |
| 3. `bisect` | `bisect_left`/`bisect_right` | `O(log n)` | `O(1)` | cách nộp bài chuẩn cho follow-up |
| 4. Tự viết nhị phân | `lower_bound(0)`, `lower_bound(1)` | `O(log n)` | `O(1)` | khi không được dùng thư viện |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().maximumCount([-1, 0, 0, 0, 2, 3])` bằng bao nhiêu?

**Bài 2 (tính tay).** Với `nums = [-4, -4, -2, 0, 1, 1, 1, 1]`, tìm số âm, số dương và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `count_equal(nums, x)` đếm số lần `x` xuất hiện trong mảng đã sắp xếp bằng `bisect`.

In [29]:
def count_equal(nums, x):
    # TODO: trả về bisect_right(nums, x) - bisect_left(nums, x)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `count_in_range(nums, lo, hi)` đếm số phần tử `x` thỏa `lo <= x <= hi` trong mảng đã sắp xếp.

**Bài 5 (biến thể / mini challenge).** Viết `maximum_count_unsorted(nums)` cho mảng **chưa sắp xếp** (chỉ dùng đếm), rồi kiểm tra kết quả bằng cách sắp xếp rồi gọi `Solution().maximumCount`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: một số âm, hai số dương (`2` và `3`), ba số `0`; đáp án `2`.
- **Mức 2 (vừa):** Bài 2: âm `3`, dương `4`, đáp án `4`. Bài 3: `bisect_right - bisect_left`.
- **Mức 3 (sâu):** Bài 4: `bisect_right(nums, hi) - bisect_left(nums, lo)`. Bài 5: đếm `neg`, `pos` rồi `max`; sắp xếp không đổi số lượng nên kết quả giống.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert Solution().maximumCount([-1, 0, 0, 0, 2, 3]) == 2

# Bài 2
nums2 = [-4, -4, -2, 0, 1, 1, 1, 1]
assert bisect_left(nums2, 0) == 3 and len(nums2) - bisect_right(nums2, 0) == 4
assert Solution().maximumCount(nums2) == 4

# Bài 3
def count_equal(nums, x):
    return bisect_right(nums, x) - bisect_left(nums, x)

# Bài 4
def count_in_range(nums, lo, hi):
    return bisect_right(nums, hi) - bisect_left(nums, lo)

# Bài 5
def maximum_count_unsorted(nums):
    neg = sum(1 for x in nums if x < 0)
    pos = sum(1 for x in nums if x > 0)
    return max(neg, pos)

assert count_equal([1, 1, 2, 2, 2, 5], 2) == 3
assert count_equal([1, 1, 2, 2, 2, 5], 3) == 0
assert count_equal([0, 0], 0) == 2
assert count_in_range([1, 2, 3, 4, 5, 6], 2, 5) == 4
assert count_in_range([1, 2, 3], 10, 20) == 0
assert count_in_range([-3, -1, 0, 4], -1, 0) == 2
random.seed(29)
for _ in range(300):
    arr = gen(random.randrange(5))
    shuffled = arr[:]
    random.shuffle(shuffled)
    assert maximum_count_unsorted(shuffled) == Solution().maximumCount(arr)
    assert count_equal(arr, 0) == arr.count(0)
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mảng sắp xếp: đếm theo ranh giới bằng `bisect`)

```python
class Solution:
    def maximumCount(self, nums: List[int]) -> int:
        neg = bisect_left(nums, 0)                    # số phần tử < 0
        pos = len(nums) - bisect_right(nums, 0)       # số phần tử > 0
        return max(neg, pos)
```

**Checklist**

- [ ] Số `0` **không** tính vào số dương hay số âm (`bisect_left` cho âm, `bisect_right` cho dương)?
- [ ] Trả về `max(neg, pos)`, không phải tổng?
- [ ] Đã thử mảng toàn âm, toàn dương, toàn `0`, và có số `0` ở giữa?
- [ ] Tự viết nhị phân thì dùng khoảng nửa mở `[0, n)` (`hi = n`)?
- [ ] Đã nhớ `bisect_left` = số phần tử `< x`, `bisect_right` = số phần tử `<= x`?
- [ ] Đã import `bisect_left`, `bisect_right` và `from typing import List`?
- [ ] Signature đúng `maximumCount(self, nums: List[int]) -> int`?